# Experiment review — feature importance & holdout metrics

Discovers experiments on R2, shows a catalog, then reviews the one you pick (feature importance, training manifest / grid search, holdout metrics).

Reads from v2 layout:
- `model_bank/experiments/{exp_id}/feature_importance/`
- `model_bank/experiments/{exp_id}/manifest/`
- `model_bank/experiments/{exp_id}/metrics/`
- `gold/model_predictions/prediction_date={date}/{exp_id}/`

**Selection:** run the catalog cell, set `SELECT_ROW` or `SELECT_EXP_ID`, then run the rest.

In [1]:
import json
import os
import pickle
import re
import sys
import tempfile
from pathlib import Path
from typing import Any

import pandas as pd
import yaml
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "utils" / "spark_session.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

GOLD_DIR = PROJECT_ROOT / "include" / "gold"
if str(GOLD_DIR) not in sys.path:
    sys.path.insert(0, str(GOLD_DIR))

load_dotenv(PROJECT_ROOT / ".env")
os.environ.setdefault(
    "PYSPARK_SUBMIT_ARGS",
    "--driver-memory 4g --executor-memory 4g pyspark-shell",
)

import pyspark.sql.functions as F

from run_paths import (
    load_run_registry,
    model_bank_experiment_root,
    model_bank_experiment_subdir,
    prediction_delta_path,
    model_bank_prediction_metrics_manifest_path,
    model_bank_threshold_sweep_path,
)
from utils.spark_session import create_spark_session

INCLUDE_DIR = PROJECT_ROOT / "include"
if str(INCLUDE_DIR) not in sys.path:
    sys.path.insert(0, str(INCLUDE_DIR))

from model_pipeline.multilabel_core import (  # noqa: E402
    DEFAULT_THRESHOLD_SWEEP,
    PREDICTED_LABELS_COL,
    TARGET_LABELS_COL,
    apply_multilabel_threshold,
    compute_threshold_sweep,
    evaluate_multilabel,
    label_prob_column_map,
    prob_columns_in_df,
)

with open(PROJECT_ROOT / "schema.yaml") as f:
    schema = yaml.safe_load(f)

spark = create_spark_session("experiment-review")
print(f"Project root: {PROJECT_ROOT}")
print(f"Spark: {spark.version}")

Picked up JAVA_TOOL_OPTIONS: --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED
Picked up JAVA_TOOL_OPTIONS: --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL

:: loading settings :: url = jar:file:/usr/local/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d0fae20d-2941-4aa7-84ae-129eea6a94ff;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.2.1/delta-spark_2.12-3.2.1.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.2.1!delta-spark_2.12.jar (370ms)
downloading https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar ...
	[SUCCESSFU

Project root: /app
Spark: 3.5.4


In [2]:
BUCKET_ROOT = f"s3a://{schema['r2']['bucket']}"


def list_r2_tree(spark, path: str, max_depth: int = 2, max_entries: int = 30) -> None:
    """Print folders/files under an s3a:// path (R2)."""
    sc = spark.sparkContext
    URI = sc._jvm.java.net.URI
    Path = sc._jvm.org.apache.hadoop.fs.Path
    FileSystem = sc._jvm.org.apache.hadoop.fs.FileSystem

    fs = FileSystem.get(URI(path), sc._jsc.hadoopConfiguration())

    def _walk(current: str, depth: int) -> None:
        p = Path(current)
        if not fs.exists(p):
            print(f"{'  ' * depth}(not found) {current}")
            return

        statuses = list(fs.listStatus(p))
        statuses.sort(key=lambda s: (not s.isDirectory(), s.getPath().getName().lower()))

        shown = 0
        for status in statuses:
            if shown >= max_entries:
                remaining = len(statuses) - shown
                print(f"{'  ' * (depth + 1)}... and {remaining} more")
                break

            name = status.getPath().getName()
            full = status.getPath().toString()
            prefix = "  " * depth

            if status.isDirectory():
                print(f"{prefix}{name}/")
                if depth < max_depth:
                    _walk(full, depth + 1)
            else:
                size_mb = status.getLen() / (1024 * 1024)
                print(f"{prefix}{name}  ({size_mb:.2f} MB)")

            shown += 1

    print(path)
    _walk(path, 0)


# Medallion roots from schema.yaml
LAYER_PATHS = {
    "landing": schema["landing"]["path"],
    "bronze": schema["bronze"]["path"],
    "silver": schema["silver"]["path"],
    "gold": schema["gold"]["path"],
    "model_bank": schema["model_bank"]["path"], 

}

print(f"Bucket root: {BUCKET_ROOT}\n")
for layer, path in LAYER_PATHS.items():
    print("=" * 70)
    print(layer)
    print("=" * 70)
    list_r2_tree(spark, path, max_depth=2, max_entries=25)
    print()

# Drill into a specific table (change path / depth as needed)
# list_r2_tree(spark, f"{schema['gold']['path']}/{schema['gold']['tables']['ngram_count']['path']}", max_depth=2)

Bucket root: s3a://cs611-project

landing
s3a://cs611-project/landing
legal_docs/
  drift_data/
    EurLex_snapshot_20050101.csv  (19.79 MB)
    EurLex_snapshot_20060101.csv  (28.20 MB)
    EurLex_snapshot_20070101.csv  (24.92 MB)
    EurLex_snapshot_20080101.csv  (34.70 MB)
    EurLex_snapshot_20090101.csv  (30.85 MB)
    EurLex_snapshot_20100101.csv  (22.78 MB)
    EurLex_snapshot_20110101.csv  (29.10 MB)
    EurLex_snapshot_20120101.csv  (25.94 MB)
    EurLex_snapshot_20130101.csv  (34.31 MB)
    EurLex_snapshot_20140101.csv  (47.16 MB)
    EurLex_snapshot_20150101.csv  (35.35 MB)
    EurLex_snapshot_20160101.csv  (40.48 MB)
    EurLex_snapshot_20170101.csv  (37.47 MB)
    EurLex_snapshot_20180101.csv  (32.65 MB)
    EurLex_snapshot_20190101.csv  (24.23 MB)
  EurLex_snapshot_19870101.csv  (6.44 MB)
  EurLex_snapshot_19880101.csv  (8.06 MB)
  EurLex_snapshot_19890101.csv  (7.17 MB)
  EurLex_snapshot_19900101.csv  (8.83 MB)
  EurLex_snapshot_19910101.csv  (9.64 MB)
  EurLex_snapshot_1

In [3]:
def load_bytes_from_path(path: str, spark) -> bytes:
    jvm = spark.sparkContext._jvm
    fd, local_path = tempfile.mkstemp()
    os.close(fd)
    try:
        hadoop_path = jvm.org.apache.hadoop.fs.Path(path)
        fs = hadoop_path.getFileSystem(spark.sparkContext._jsc.hadoopConfiguration())
        if not fs.exists(hadoop_path):
            raise FileNotFoundError(path)
        fs.copyToLocalFile(False, hadoop_path, jvm.org.apache.hadoop.fs.Path(local_path))
        with open(local_path, "rb") as handle:
            return handle.read()
    finally:
        if os.path.exists(local_path):
            os.unlink(local_path)


def load_pickle(path: str, spark) -> Any:
    return pickle.loads(load_bytes_from_path(path, spark))


def load_json(path: str, spark) -> Any:
    return json.loads(load_bytes_from_path(path, spark).decode("utf-8"))


def path_exists(spark, path: str) -> bool:
    sc = spark.sparkContext
    URI = sc._jvm.java.net.URI
    Path = sc._jvm.org.apache.hadoop.fs.Path
    FileSystem = sc._jvm.org.apache.hadoop.fs.FileSystem
    fs = FileSystem.get(URI(path), sc._jsc.hadoopConfiguration())
    return fs.exists(Path(path))


def list_child_names(spark, base: str) -> list[str]:
    sc = spark.sparkContext
    URI = sc._jvm.java.net.URI
    Path = sc._jvm.org.apache.hadoop.fs.Path
    FileSystem = sc._jvm.org.apache.hadoop.fs.FileSystem
    fs = FileSystem.get(URI(base), sc._jsc.hadoopConfiguration())
    p = Path(base)
    if not fs.exists(p):
        return []
    return sorted(st.getPath().getName() for st in fs.listStatus(p))


def latest_matching_dirs(dirs: list[str], prefix: str, suffix: str) -> str | None:
    for base in dirs:
        matches = [n for n in list_child_names(spark, base) if n.startswith(prefix) and n.endswith(suffix)]
        if matches:
            return f"{base}/{sorted(matches)[-1]}"
    return None


def label_universe_from_predictions(predictions) -> list[str]:
    labels = (
        predictions.select(F.explode(F.col(TARGET_LABELS_COL)).alias("lbl"))
        .union(predictions.select(F.explode(F.col(PREDICTED_LABELS_COL)).alias("lbl")))
        .select("lbl")
        .distinct()
        .orderBy("lbl")
    )
    return [row.lbl for row in labels.collect()]


_MANIFEST_RE = re.compile(r"^(logistic_regression|random_forest)_(\d{8})\.pkl$")
_METRICS_RE = re.compile(r"^prediction_(\d{8})\.pkl$")
_SWEEP_RE = re.compile(r"^threshold_sweep_(\d{8})\.pkl$")
_FI_RE = re.compile(r"^feature_importance_(\d{8})\.json$")


def _yyyymmdd_to_iso(date_str: str) -> str:
    if re.fullmatch(r"\d{8}", date_str):
        return f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
    return date_str


def discover_experiments(spark) -> pd.DataFrame:
    """Scan R2 for experiments, manifests, predictions, and metrics."""
    registry = load_run_registry()
    mb = schema["model_bank"]
    experiments_base = f"{mb['path']}/{mb['experiments']['base']}"
    predictions_base = f"{schema['gold']['path']}/{schema['gold']['model_predictions']['base']}"
    date_prefix = schema["gold"]["model_predictions"]["date_partition_prefix"]

    exp_ids = sorted(
        name for name in list_child_names(spark, experiments_base) if name.startswith("exp")
    )

    prediction_partitions = [
        part[len(date_prefix) :]
        for part in list_child_names(spark, predictions_base)
        if part.startswith(date_prefix)
    ]

    rows: list[dict[str, Any]] = []
    for exp_id in exp_ids:
        meta = registry.get("experiments", {}).get(exp_id, {})
        exp_root = f"{experiments_base}/{exp_id}"
        manifest_dirs = [f"{exp_root}/manifest", f"{exp_root}/model"]
        fi_dirs = [
            f"{exp_root}/{mb['experiments']['feature_importance_dir']}",
            f"{exp_root}/{mb['experiments']['model_dir']}",
        ]
        metrics_dir = f"{exp_root}/{mb['experiments']['metrics_dir']}"

        manifests: list[tuple[str, str, str, str]] = []
        for directory in manifest_dirs:
            for name in list_child_names(spark, directory):
                match = _MANIFEST_RE.match(name)
                if match:
                    manifests.append((directory, name, match.group(1), match.group(2)))
        latest_manifest = sorted(manifests, key=lambda item: item[3])[-1] if manifests else None

        pred_dates = sorted(
            date_str
            for date_str in prediction_partitions
            if path_exists(spark, f"{predictions_base}/{date_prefix}{date_str}/{exp_id}")
        )

        metric_dates = sorted(
            {
                match.group(1)
                for name in list_child_names(spark, metrics_dir)
                if (match := _METRICS_RE.match(name))
            }
        )

        sweep_dates = sorted(
            {
                match.group(1)
                for name in list_child_names(spark, metrics_dir)
                if (match := _SWEEP_RE.match(name))
            }
        )

        fi_dates = sorted(
            {
                match.group(1)
                for directory in fi_dirs
                for name in list_child_names(spark, directory)
                if (match := _FI_RE.match(name))
            }
        )

        rows.append(
            {
                "exp_id": exp_id,
                "model_type": meta.get("model_type") or (latest_manifest[2] if latest_manifest else ""),
                "feature_set": meta.get("feature_set", ""),
                "grid_search": meta.get("grid_search"),
                "train_date": latest_manifest[3] if latest_manifest else None,
                "manifest_prefix": latest_manifest[2] if latest_manifest else None,
                "train_manifest": (
                    f"{latest_manifest[0]}/{latest_manifest[1]}" if latest_manifest else None
                ),
                "prediction_dates": ", ".join(pred_dates) or "—",
                "latest_prediction_date": pred_dates[-1] if pred_dates else None,
                "metrics_dates": ", ".join(_yyyymmdd_to_iso(d) for d in metric_dates) or "—",
                "threshold_sweep_dates": ", ".join(_yyyymmdd_to_iso(d) for d in sweep_dates) or "—",
                "feature_importance_date": fi_dates[-1] if fi_dates else None,
            }
        )

    return pd.DataFrame(rows)

In [4]:
catalog = discover_experiments(spark).reset_index(names="row")
display(catalog)

,row,exp_id,model_type,feature_set,grid_search,train_date,manifest_prefix,train_manifest,prediction_dates,latest_prediction_date,metrics_dates,threshold_sweep_dates,feature_importance_date
0,0,exp001_RF_emb_tfidf_dcw,random_forest,tfidf_dcw_embeddings,False,20260612,random_forest,s3a://cs611-project/model_bank/experiments/exp...,2026-06-12,2026-06-12,2026-06-12,2026-06-12,20260617
1,1,exp002_LR_emb_tfidf_dcw,logistic_regression,tfidf_dcw_embeddings,False,20260612,logistic_regression,s3a://cs611-project/model_bank/experiments/exp...,2026-06-13,2026-06-13,2026-06-13,2026-06-13,20260617
2,2,exp003_LR_tfidf_dcw,logistic_regression,tfidf_dcw,False,20260613,logistic_regression,s3a://cs611-project/model_bank/experiments/exp...,2026-06-13,2026-06-13,2026-06-13,2026-06-13,20260617
3,3,exp004_LR_tfidf_dcw_gs,logistic_regression,tfidf_dcw,True,20260615,logistic_regression,s3a://cs611-project/model_bank/experiments/exp...,2026-06-15,2026-06-15,2026-06-15,2026-06-15,20260615
4,4,exp005_RF_tfidf_dcw_gs,random_forest,tfidf_dcw,False,20260615,random_forest,s3a://cs611-project/model_bank/experiments/exp...,2026-06-16,2026-06-16,2026-06-16,2026-06-16,20260615
5,5,exp006_LR_tfidf,logistic_regression,,None,20260617,logistic_regression,s3a://cs611-project/model_bank/experiments/exp...,2026-06-17,2026-06-17,2026-06-17,2026-06-17,20260617


In [5]:
# --- pick what to review (after scanning the catalog below) ---
SELECT_ROW = 5             # 0-based row index in the table; None = use SELECT_EXP_ID or last row
SELECT_EXP_ID = None       # e.g. "exp004_LR_tfidf_dcw_gs"
SELECT_PREDICTION_DATE = None  # YYYY-MM-DD; None = latest_prediction_date for chosen exp

if catalog.empty:
    raise RuntimeError("No experiments found under model_bank/experiments/ on R2.")

if SELECT_EXP_ID:
    matches = catalog[catalog["exp_id"] == SELECT_EXP_ID]
    if matches.empty:
        raise ValueError(f"Unknown exp_id: {SELECT_EXP_ID!r}")
    row = matches.iloc[0]
elif SELECT_ROW is not None:
    row = catalog.iloc[int(SELECT_ROW)]
else:
    row = catalog.iloc[-1]

EXP_ID = row["exp_id"]
PREDICTION_DATE = SELECT_PREDICTION_DATE or row["latest_prediction_date"]
MODEL_MANIFEST_PREFIX = row["manifest_prefix"]
TRAIN_DATE = row["train_date"]
TRAIN_MANIFEST_PATH = row["train_manifest"]

EXPERIMENT_ROOT = model_bank_experiment_root(EXP_ID)
FI_DIRS = [
    model_bank_experiment_subdir(EXP_ID, "feature_importance_dir"),
    f"{model_bank_experiment_subdir(EXP_ID, 'model_dir')}",
]
MANIFEST_DIRS = [
    model_bank_experiment_subdir(EXP_ID, "manifest_dir"),
    model_bank_experiment_subdir(EXP_ID, "model_dir"),
]

PRED_DELTA = prediction_delta_path(PREDICTION_DATE, EXP_ID) if PREDICTION_DATE else None
PRED_METRICS_PKL = (
    model_bank_prediction_metrics_manifest_path(EXP_ID, PREDICTION_DATE)
    if PREDICTION_DATE
    else None
)
THRESHOLD_SWEEP_PKL = (
    model_bank_threshold_sweep_path(EXP_ID, PREDICTION_DATE)
    if PREDICTION_DATE
    else None
)

print(f"Selected row {int(row['row'])}: {EXP_ID}")
print("Experiment:", EXPERIMENT_ROOT)
if PREDICTION_DATE:
    print("Predictions:", PRED_DELTA)
    print("Metrics manifest:", PRED_METRICS_PKL)
    print("Threshold sweep:", THRESHOLD_SWEEP_PKL)
else:
    print("Predictions: (none on R2 for this exp)")
if TRAIN_MANIFEST_PATH:
    print("Training manifest:", TRAIN_MANIFEST_PATH)
else:
    print("Training manifest: (none found)")

Selected row 5: exp006_LR_tfidf
Experiment: s3a://cs611-project/model_bank/experiments/exp006_LR_tfidf
Predictions: s3a://cs611-project/gold/model_predictions/prediction_date=2026-06-17/exp006_LR_tfidf
Metrics manifest: s3a://cs611-project/model_bank/experiments/exp006_LR_tfidf/metrics/prediction_20260617.pkl
Threshold sweep: s3a://cs611-project/model_bank/experiments/exp006_LR_tfidf/metrics/threshold_sweep_20260617.pkl
Training manifest: s3a://cs611-project/model_bank/experiments/exp006_LR_tfidf/manifest/logistic_regression_20260617.pkl


## Feature importance (LR: |coefficient|; RF: Gini importance)

In [6]:
fi_path = latest_matching_dirs(FI_DIRS, "feature_importance_", ".json")
if fi_path is None:
    print(f"No feature_importance_*.json for {EXP_ID}.")
    print("Run: python scripts/batch_evaluate_experiments.py")
else:
    fi = load_json(fi_path, spark)
    print(f"Loaded: {fi_path}")
    print(f"Method: {fi.get('method')} | top_k={fi.get('top_k')} | labels={fi.get('num_labels')}")
    global_df = pd.DataFrame(fi["global_top"])
    display(global_df)

Loaded: s3a://cs611-project/model_bank/experiments/exp006_LR_tfidf/feature_importance/feature_importance_20260617.json
Method: abs_logistic_regression_coefficient | top_k=50 | labels=15


,feature,mean_abs_coefficient,rank
0,tfidf:done,835.156570,1
1,tfidf:regard treaty,715.265086,2
2,tfidf:regard treaty establishing,467.221923,3
3,tfidf:official,357.334915,4
4,tfidf:official journal,317.180864,5
5,tfidf:regard,256.425671,6
6,tfidf:european,197.027188,7
7,tfidf:treaty establishing,166.763825,8
8,tfidf:done brussels,102.059056,9
9,tfidf:article,91.763016,10


In [7]:
LABEL_TO_SHOW = sorted(fi["per_label"].keys())[0] if fi.get("per_label") else None

if LABEL_TO_SHOW:
    label_df = pd.DataFrame(fi["per_label"][LABEL_TO_SHOW])
    print(f"Per-label top features: {LABEL_TO_SHOW!r}")
    display(label_df)
else:
    print("No per-label block in feature importance JSON.")

Per-label top features: 'agriculture & food safety'


,abs_coefficient,coefficient,feature,rank
0,445.149588,445.149588,tfidf:done,1
1,248.515212,248.515212,tfidf:regard treaty,2
2,213.683914,213.683914,tfidf:regard treaty establishing,3
3,207.988351,-207.988351,tfidf:european,4
4,109.131765,-109.131765,tfidf:official journal,5
5,93.993861,-93.993861,tfidf:official,6
6,74.950062,74.950062,tfidf:treaty establishing european,7
7,64.511628,-64.511628,tfidf:article,8
8,62.848867,62.848867,tfidf:treaty establishing,9
9,61.866028,61.866028,tfidf:done brussels,10


## Training manifest & grid-search trials

In [8]:
manifest_path = TRAIN_MANIFEST_PATH
if manifest_path is None and MODEL_MANIFEST_PREFIX:
    if TRAIN_DATE:
        manifest_path = f"{MANIFEST_DIRS[0]}/{MODEL_MANIFEST_PREFIX}_{TRAIN_DATE}.pkl"
    else:
        manifest_path = latest_matching_dirs(MANIFEST_DIRS, f"{MODEL_MANIFEST_PREFIX}_", ".pkl")

if manifest_path is None:
    raise FileNotFoundError(f"No training manifest found for {EXP_ID}")

train_manifest = load_pickle(manifest_path, spark)
print(f"Training manifest: {manifest_path}")
print("Best hyperparameters:", train_manifest.get("hyperparameters"))

if "grid_search" in train_manifest:
    gs = train_manifest["grid_search"]
    trials_df = pd.DataFrame(gs["trials"])
    metric = gs.get("metric", "micro_f1")
    print(f"Grid search metric: {metric}")
    display(trials_df.sort_values(metric, ascending=False))
else:
    print("No grid_search block in training manifest.")

Training manifest: s3a://cs611-project/model_bank/experiments/exp006_LR_tfidf/manifest/logistic_regression_20260617.pkl
Best hyperparameters: {'maxIter': 100, 'regParam': 0.0, 'elasticNetParam': 0.0, 'model_type': 'logistic_regression', 'multilabel_threshold': 0.5, 'max_labels': None, 'train_only': True, 'grid_search': False}
No grid_search block in training manifest.


## Holdout prediction metrics

In [9]:
def metrics_to_rows(metrics: dict[str, Any]) -> pd.DataFrame:
    rows = []
    for split, values in metrics.items():
        if not isinstance(values, dict):
            continue
        row = {"split": split, **values}
        rows.append(row)
    return pd.DataFrame(rows)


def load_metrics_from_delta(delta_path: str) -> dict[str, Any]:
    print(f"Computing metrics from Delta: {delta_path}")
    predictions = spark.read.format("delta").load(delta_path)
    label_list = label_universe_from_predictions(predictions)
    print(f"Label universe: {len(label_list)} labels")
    return evaluate_multilabel(predictions, label_list)


metrics_source = None
metrics = None

if PRED_METRICS_PKL:
    try:
        eval_manifest = load_pickle(PRED_METRICS_PKL, spark)
        metrics = eval_manifest["metrics"]
        metrics_source = PRED_METRICS_PKL
    except FileNotFoundError:
        pass

    if metrics is None:
        json_path = PRED_METRICS_PKL.replace(".pkl", ".json")
        try:
            eval_manifest = load_json(json_path, spark)
            metrics = eval_manifest["metrics"]
            metrics_source = json_path
        except FileNotFoundError:
            pass

if metrics is None and PRED_DELTA:
    try:
        metrics = load_metrics_from_delta(PRED_DELTA)
        metrics_source = PRED_DELTA
    except FileNotFoundError as exc:
        print(f"No metrics manifest at {PRED_METRICS_PKL}")
        print(f"Delta not found: {exc}")
        print("Run: --predict-only --predict-stage predict (then optionally metrics)")
elif metrics is None:
    print(f"No predictions on R2 for {EXP_ID}.")
    print("Run: --predict-only --predict-stage predict --prediction-date YYYY-MM-DD")

if metrics is not None:
    print(f"Metrics source: {metrics_source}")
    display(metrics_to_rows(metrics))

Metrics source: s3a://cs611-project/model_bank/experiments/exp006_LR_tfidf/metrics/prediction_20260617.pkl


,split,documents,exact_match_ratio,micro_precision,micro_recall,micro_f1,macro_precision,macro_recall,macro_f1,hamming_loss
0,holdout_overall,9693,0.435056,0.782422,0.859433,0.819122,0.660995,0.808869,0.706167,0.100973
1,holdout_val,5338,0.455789,0.796629,0.865121,0.829464,0.670768,0.814838,0.713051,0.094667
2,holdout_test,2668,0.475637,0.804134,0.868236,0.834957,0.676696,0.811936,0.718849,0.091454
3,holdout_oot,1687,0.305276,0.708541,0.827370,0.763359,0.613721,0.782648,0.662869,0.135981


In [10]:
# Threshold sweep — loads from R2 if saved, else computes from prob_* on the Delta.

SWEEP_SPLITS = ("holdout_overall", "holdout_val", "holdout_test", "holdout_oot")
METRIC_COLS = (
    "micro_f1", "macro_f1", "exact_match_ratio",
    "micro_precision", "micro_recall", "macro_precision", "macro_recall",
    "hamming_loss", "documents",
)


def sweep_rows_to_df(rows: list[dict[str, Any]], *, split: str = "holdout_overall") -> pd.DataFrame:
    filtered = [r for r in rows if r.get("split") == split]
    return pd.DataFrame(filtered).sort_values("threshold")


if not PRED_DELTA:
    print(f"No prediction Delta for {EXP_ID}.")
elif THRESHOLD_SWEEP_PKL and path_exists(spark, THRESHOLD_SWEEP_PKL):
    sweep = load_pickle(THRESHOLD_SWEEP_PKL, spark)
    print(f"Loaded threshold sweep from {THRESHOLD_SWEEP_PKL}")
    for split in SWEEP_SPLITS:
        df = sweep_rows_to_df(sweep["rows"], split=split)
        if not df.empty:
            print(f"\n{split}")
            display(df[["threshold", *METRIC_COLS]])
else:
    predictions = spark.read.format("delta").load(PRED_DELTA)
    if not prob_columns_in_df(predictions):
        print(f"No prob_* on {PRED_DELTA}. Run: python scripts/batch_evaluate_experiments.py")
    else:
        label_list = label_universe_from_predictions(predictions)
        sweep = compute_threshold_sweep(predictions, label_list, DEFAULT_THRESHOLD_SWEEP)
        print("Computed on the fly — run batch_evaluate_experiments.py to save to R2.")
        for split in SWEEP_SPLITS:
            df = sweep_rows_to_df(sweep["rows"], split=split)
            if not df.empty:
                print(f"\n{split}")
                display(df[["threshold", *METRIC_COLS]])

Loaded threshold sweep from s3a://cs611-project/model_bank/experiments/exp006_LR_tfidf/metrics/threshold_sweep_20260617.pkl

holdout_overall


,threshold,micro_f1,macro_f1,exact_match_ratio,micro_precision,micro_recall,macro_precision,macro_recall,hamming_loss,documents
0,0.30,0.819224,0.705805,0.432683,0.780286,0.862251,0.659172,0.811047,0.101235,9693
1,0.35,0.819161,0.705852,0.433406,0.780808,0.861475,0.659604,0.810486,0.101186,9693
2,0.40,0.819243,0.706018,0.434231,0.781383,0.860958,0.660067,0.810184,0.101070,9693
3,0.45,0.819212,0.706268,0.434540,0.782009,0.860131,0.660709,0.809702,0.100994,9693
4,0.50,0.819122,0.706167,0.435056,0.782422,0.859433,0.660995,0.808869,0.100973,9693
5,0.55,0.819122,0.706329,0.434437,0.783003,0.858735,0.661536,0.808374,0.100891,9693
6,0.60,0.819111,0.706452,0.434025,0.783477,0.858140,0.661890,0.807990,0.100829,9693
7,0.65,0.819153,0.706783,0.434540,0.784181,0.857390,0.662651,0.807574,0.100712,9693
8,0.70,0.819024,0.706648,0.434747,0.784767,0.856408,0.663024,0.806585,0.100684,9693
9,0.75,0.819076,0.706961,0.435056,0.785691,0.855425,0.663875,0.805994,0.100533,9693



holdout_val


,threshold,micro_f1,macro_f1,exact_match_ratio,micro_precision,micro_recall,macro_precision,macro_recall,hamming_loss,documents
0,0.30,0.829393,0.712573,0.453353,0.794363,0.867655,0.668759,0.816886,0.094992,5338
1,0.35,0.829379,0.712712,0.454290,0.794888,0.866998,0.669231,0.816549,0.094929,5338
2,0.40,0.829544,0.712886,0.455227,0.795588,0.866529,0.669768,0.816181,0.094767,5338
3,0.45,0.829478,0.713067,0.455227,0.796099,0.865778,0.670295,0.815853,0.094730,5338
4,0.50,0.829464,0.713051,0.455789,0.796629,0.865121,0.670768,0.814838,0.094667,5338
5,0.55,0.829392,0.713179,0.454665,0.797255,0.864229,0.671345,0.814184,0.094617,5338
6,0.60,0.829404,0.713356,0.454290,0.797996,0.863385,0.671897,0.813576,0.094517,5338
7,0.65,0.829517,0.713739,0.455414,0.798688,0.862821,0.672692,0.813362,0.094380,5338
8,0.70,0.829427,0.713585,0.455789,0.799286,0.861930,0.673055,0.812382,0.094342,5338
9,0.75,0.829445,0.713941,0.456351,0.800209,0.860897,0.673958,0.811793,0.094218,5338



holdout_test


,threshold,micro_f1,macro_f1,exact_match_ratio,micro_precision,micro_recall,macro_precision,macro_recall,hamming_loss,documents
0,0.30,0.835438,0.718786,0.473013,0.802227,0.871518,0.675074,0.814174,0.091479,2668
1,0.35,0.835253,0.718654,0.473763,0.802681,0.870581,0.675386,0.813425,0.091504,2668
2,0.40,0.835111,0.718710,0.474138,0.802978,0.869924,0.675689,0.813175,0.091529,2668
3,0.45,0.834918,0.718844,0.474888,0.803661,0.868705,0.676477,0.812201,0.091529,2668
4,0.50,0.834957,0.718849,0.475637,0.804134,0.868236,0.676696,0.811936,0.091454,2668
5,0.55,0.835010,0.718921,0.474513,0.804556,0.867861,0.677033,0.811620,0.091379,2668
6,0.60,0.834860,0.718955,0.473388,0.804681,0.867392,0.677184,0.811441,0.091429,2668
7,0.65,0.834794,0.719194,0.473388,0.805122,0.866735,0.677789,0.811192,0.091404,2668
8,0.70,0.834547,0.718945,0.473388,0.805798,0.865422,0.678286,0.809804,0.091429,2668
9,0.75,0.834510,0.719032,0.473388,0.806545,0.864485,0.678874,0.809274,0.091354,2668



holdout_oot


,threshold,micro_f1,macro_f1,exact_match_ratio,micro_precision,micro_recall,macro_precision,macro_recall,hamming_loss,documents
0,0.30,0.763432,0.662589,0.303497,0.706494,0.830352,0.612357,0.785214,0.136416,1687
1,0.35,0.763394,0.662591,0.303497,0.707078,0.829457,0.612753,0.784284,0.136297,1687
2,0.40,0.763559,0.662879,0.304683,0.707686,0.829010,0.613212,0.784109,0.136100,1687
3,0.45,0.763881,0.663427,0.305276,0.708567,0.828563,0.613930,0.783934,0.135783,1687
4,0.50,0.763359,0.662869,0.305276,0.708541,0.827370,0.613721,0.782648,0.135981,1687
5,0.55,0.763491,0.663241,0.307054,0.709207,0.826774,0.614359,0.782415,0.135783,1687
6,0.60,0.763649,0.663392,0.307647,0.709479,0.826774,0.614503,0.782415,0.135665,1687
7,0.65,0.763586,0.663439,0.307054,0.710472,0.825283,0.615053,0.781012,0.135467,1687
8,0.70,0.763528,0.663478,0.307054,0.710925,0.824538,0.615351,0.780515,0.135388,1687
9,0.75,0.763807,0.663877,0.307054,0.712076,0.823643,0.616200,0.779753,0.135033,1687


In [7]:
# Optional: quick sanity check on prediction Delta row count
if not PRED_DELTA:
    print("No prediction Delta for this experiment (see catalog prediction_dates).")
else:
    try:
        pred_df = spark.read.format("delta").load(PRED_DELTA)
        print(f"Prediction rows: {pred_df.count():,}")
        pred_df.groupBy("category").count().orderBy("category").show()
    except Exception as exc:
        print(f"Could not read {PRED_DELTA}: {exc}")

Prediction rows: 9,693


[Stage 366:>                                                        (0 + 1) / 1]

+--------+-----+
|category|count|
+--------+-----+
|     oot| 1687|
|    test| 2668|
|     val| 5338|
+--------+-----+

